In [1]:
from dotenv import load_dotenv
import os

result = load_dotenv()
print("load_dotenv() returned:", result)  # True aana chahiye

APP_ID = os.getenv("ADZUNA_APP_ID")
APP_KEY = os.getenv("ADZUNA_APP_KEY")
print("APP_ID:", APP_ID)
print("APP_KEY:", APP_KEY)

load_dotenv() returned: True
APP_ID: 2aedd3a8
APP_KEY: 24dd10c13e99838c8c9c4d9943ee8a31


In [2]:
"""
SkillScope - Step 1B: Adzuna Data Collection
Fetches Data Analyst job postings for major Indian cities and saves
raw, untouched JSON responses to data/raw/.

Setup:
1. pip install requests python-dotenv
2. Create a .env file in the same folder with:
     ADZUNA_APP_ID=your_app_id_here
     ADZUNA_APP_KEY=your_app_key_here
3. Run: python fetch_adzuna.py
"""

import os
import json
import time
import requests
from pathlib import Path
from datetime import datetime, timezone



In [3]:

# Adzuna's country code for India is 'in'
BASE_URL = "https://api.adzuna.com/v1/api/jobs/in/search"

# Cities to query. Adzuna uses free-text location matching via 'where'.
CITIES = [
    "Bangalore",
    "Mumbai",
    "Delhi NCR",
    "Pune",
    "Hyderabad",
    "Remote",
]

SEARCH_TERM = "Data Analyst"
RESULTS_PER_PAGE = 20     # Adzuna's max per page
MAX_PAGES_PER_CITY = 5    # 5 pages x 20 = up to 100 postings per city (adjust later)
REQUEST_DELAY_SECONDS = 1  # be polite to the API, avoid rate-limit issues

RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
def fetch_page(city: str, page: int) -> dict:
    """Fetch a single page of results for a given city."""
    url = f"{BASE_URL}/{page}"
    params = {
        "app_id": APP_ID,
        "app_key": APP_KEY,
        "results_per_page": RESULTS_PER_PAGE,
        "what": SEARCH_TERM,
        "where": city,
        "content-type": "application/json",
    }
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    return response.json()

In [5]:
def fetch_city(city: str) -> list:
    """Fetch all pages for a city until no more results or MAX_PAGES_PER_CITY hit."""
    all_results = []
    for page in range(1, MAX_PAGES_PER_CITY + 1):
        print(f"  Fetching {city} - page {page}...")
        try:
            data = fetch_page(city, page)
        except requests.exceptions.HTTPError as e:
            print(f"  HTTP error on {city} page {page}: {e}")
            break
        except requests.exceptions.RequestException as e:
            print(f"  Network error on {city} page {page}: {e}")
            break

        results = data.get("results", [])
        if not results:
            print(f"  No more results for {city} after page {page - 1}.")
            break

        all_results.extend(results)
        time.sleep(REQUEST_DELAY_SECONDS)

    return all_results

In [6]:
def save_raw(city: str, results: list):
    """Save raw results for a city as timestamped JSON, untouched."""
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    safe_city = city.lower().replace(" ", "_")
    filename = RAW_DIR / f"{safe_city}_{timestamp}.json"

    payload = {
        "city_queried": city,
        "search_term": SEARCH_TERM,
        "fetched_at_utc": timestamp,
        "result_count": len(results),
        "results": results,
    }

    with open(filename, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    print(f"  Saved {len(results)} postings -> {filename}")

In [9]:
print("DEBUG check:", repr(APP_ID), repr(APP_KEY))

def main():
    print(f"Starting SkillScope data collection: '{SEARCH_TERM}' across {len(CITIES)} cities\n")
    summary = {}

    for city in CITIES:
        print(f"City: {city}")
        results = fetch_city(city)
        save_raw(city, results)
        summary[city] = len(results)
        print()

    print("=" * 40)
    print("Collection summary:")
    total = 0
    for city, count in summary.items():
        print(f"  {city}: {count} postings")
        total += count
    print(f"  TOTAL: {total} postings")
    print("=" * 40)


if __name__ == "__main__":
    main()

DEBUG check: '2aedd3a8' '24dd10c13e99838c8c9c4d9943ee8a31'
Starting SkillScope data collection: 'Data Analyst' across 6 cities

City: Bangalore
  Fetching Bangalore - page 1...
  Fetching Bangalore - page 2...
  Fetching Bangalore - page 3...
  Fetching Bangalore - page 4...
  Fetching Bangalore - page 5...
  Saved 100 postings -> data\raw\bangalore_20260820T054552Z.json

City: Mumbai
  Fetching Mumbai - page 1...
  Fetching Mumbai - page 2...
  Fetching Mumbai - page 3...
  Fetching Mumbai - page 4...
  Fetching Mumbai - page 5...
  Saved 100 postings -> data\raw\mumbai_20260820T054604Z.json

City: Delhi NCR
  Fetching Delhi NCR - page 1...
  Fetching Delhi NCR - page 2...
  Fetching Delhi NCR - page 3...
  Fetching Delhi NCR - page 4...
  No more results for Delhi NCR after page 3.
  Saved 52 postings -> data\raw\delhi_ncr_20260820T054612Z.json

City: Pune
  Fetching Pune - page 1...
  Fetching Pune - page 2...
  Fetching Pune - page 3...
  Fetching Pune - page 4...
  Fetching Pune -

In [11]:
import json

# Saari bangalore files dhoondo, sabse naye (latest) ko sort karke uthao
raw_files = sorted(RAW_DIR.glob("bangalore_*.json"))
latest_file = raw_files[-1]   # sabse aakhri = sabse recent

print("Using file:", latest_file)

with open(latest_file, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Total results in this file:", data["result_count"])
print("\nPehli posting ka structure:\n")
print(json.dumps(data["results"][0], indent=2, ensure_ascii=False))

Using file: data\raw\bangalore_20260820T054552Z.json
Total results in this file: 100

Pehli posting ka structure:

{
  "title": "Senior Data Analyst III",
  "company": {
    "__CLASS__": "Adzuna::API::Response::Company",
    "display_name": "Elsevier"
  },
  "location": {
    "area": [
      "India",
      "Karnataka",
      "Bangalore"
    ],
    "__CLASS__": "Adzuna::API::Response::Location",
    "display_name": "Bangalore, Karnataka"
  },
  "category": {
    "label": "IT Jobs",
    "tag": "it-jobs",
    "__CLASS__": "Adzuna::API::Response::Category"
  },
  "created": "2026-08-15T00:04:09Z",
  "description": "This job is with Elsevier, an inclusive employer and a member of myGwork – the largest global platform for the LGBTQ business community. Please do not contact the recruiter directly. Senior Data Analyst III Do you like working with data and analytics to gain insight to solve problems? Do you enjoy collaborating across teams to build and deliver products that make a difference? A

In [14]:

import sqlite3
import json
from pathlib import Path
from collections import defaultdict

DB_PATH = "skillscope.db"

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# ---- Step A: Create Table  ----
cursor.execute("""
CREATE TABLE IF NOT EXISTS postings (
    id TEXT PRIMARY KEY,
    title TEXT,
    company TEXT,
    city_queried TEXT,
    location_display_name TEXT,
    category_label TEXT,
    contract_time TEXT,
    salary_min REAL,
    salary_max REAL,
    salary_is_predicted TEXT,
    created_date TEXT,
    description TEXT,
    redirect_url TEXT,
    fetched_at_utc TEXT
)
""")
conn.commit()
print("Table 'postings' ready.")

# ---- Step B: Har city ki sirf LATEST file dhoondo ----
city_to_latest_file = {}
for file in RAW_DIR.glob("*.json"):
    # filename pattern: cityname_TIMESTAMP.json -> city nikaalo
    parts = file.stem.rsplit("_", 1)  # last underscore se split -> [city, timestamp]
    city_key = parts[0]
    if city_key not in city_to_latest_file or file.name > city_to_latest_file[city_key].name:
        city_to_latest_file[city_key] = file

print("\nUsing these files (latest per city):")
for city, f in city_to_latest_file.items():
    print(f"  {city}: {f.name}")

# ---- Step C: Har file load karo aur DB me insert karo ----
total_inserted = 0

for city_key, file in city_to_latest_file.items():
    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)

    city_queried = data.get("city_queried", city_key)
    fetched_at = data.get("fetched_at_utc", "")
    results = data.get("results", [])

    rows_this_file = 0
    for job in results:
        job_id = job.get("id")
        title = job.get("title", "")
        company = (job.get("company") or {}).get("display_name", "")
        location_display = (job.get("location") or {}).get("display_name", "")
        category_label = (job.get("category") or {}).get("label", "")
        contract_time = job.get("contract_time", "")
        salary_min = job.get("salary_min")
        salary_max = job.get("salary_max")
        salary_is_predicted = job.get("salary_is_predicted", "")
        created_date = job.get("created", "")
        description = job.get("description", "")
        redirect_url = job.get("redirect_url", "")

        cursor.execute("""
            INSERT OR REPLACE INTO postings
            (id, title, company, city_queried, location_display_name, category_label,
             contract_time, salary_min, salary_max, salary_is_predicted, created_date,
             description, redirect_url, fetched_at_utc)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (job_id, title, company, city_queried, location_display, category_label,
              contract_time, salary_min, salary_max, salary_is_predicted, created_date,
              description, redirect_url, fetched_at))

        rows_this_file += 1

    conn.commit()
    total_inserted += rows_this_file
    print(f"  Loaded {rows_this_file} postings from {city_key}")

print(f"\nTotal rows in postings table (after de-dup by id): ", end="")
cursor.execute("SELECT COUNT(*) FROM postings")
print(cursor.fetchone()[0])

Table 'postings' ready.

Using these files (latest per city):
  bangalore: bangalore_20260820T054552Z.json
  delhi_ncr: delhi_ncr_20260820T054612Z.json
  hyderabad: hyderabad_20260820T054636Z.json
  mumbai: mumbai_20260820T054604Z.json
  pune: pune_20260820T054624Z.json
  remote: remote_20260820T054637Z.json
  Loaded 100 postings from bangalore
  Loaded 52 postings from delhi_ncr
  Loaded 100 postings from hyderabad
  Loaded 100 postings from mumbai
  Loaded 100 postings from pune
  Loaded 0 postings from remote

Total rows in postings table (after de-dup by id): 452


In [15]:
import re

# ---- Skill keyword list (Data Analyst role ke according) ----
# Key = skill ka clean naam jo hum store karenge
# Value = list of regex patterns jo description me dhoondhne hain (case-insensitive)
SKILL_PATTERNS = {
    "SQL":            [r"\bsql\b"],
    "Python":         [r"\bpython\b"],
    "R":              [r"\br programming\b", r"\br studio\b", r"\b(?<![a-z])r(?![a-z])\b.{0,15}\bprogramming\b"],
    "Excel":          [r"\bexcel\b", r"\bms excel\b"],
    "Power BI":       [r"\bpower\s?bi\b"],
    "Tableau":        [r"\btableau\b"],
    "Looker":         [r"\blooker\b"],
    "SAS":            [r"\bsas\b"],
    "Statistics":     [r"\bstatistic(s|al)?\b"],
    "Machine Learning": [r"\bmachine learning\b", r"\bml\b"],
    "AWS":            [r"\baws\b", r"\bamazon web services\b"],
    "Azure":          [r"\bazure\b"],
    "GCP":            [r"\bgcp\b", r"\bgoogle cloud\b"],
    "ETL":            [r"\betl\b"],
    "Data Warehousing": [r"\bdata warehous\w*\b"],
    "Big Data":       [r"\bbig data\b", r"\bhadoop\b", r"\bspark\b"],
    "Power Query":    [r"\bpower query\b"],
    "VBA":            [r"\bvba\b"],
    "PowerPoint":     [r"\bpowerpoint\b"],
    "A/B Testing":    [r"\ba/b test\w*\b", r"\bab test\w*\b"],
    "Google Analytics": [r"\bgoogle analytics\b"],
    "Communication":  [r"\bcommunication skills\b"],
}

# Compile all patterns once for speed
COMPILED_PATTERNS = {
    skill: [re.compile(p, re.IGNORECASE) for p in patterns]
    for skill, patterns in SKILL_PATTERNS.items()
}


def extract_skills(description: str) -> list:
    """Return list of skill names found in the description text."""
    if not description:
        return []
    found = []
    for skill, patterns in COMPILED_PATTERNS.items():
        if any(p.search(description) for p in patterns):
            found.append(skill)
    return found


# ---- Table banao skill mapping ke liye ----
cursor.execute("""
CREATE TABLE IF NOT EXISTS posting_skills (
    posting_id TEXT,
    skill TEXT,
    PRIMARY KEY (posting_id, skill),
    FOREIGN KEY (posting_id) REFERENCES postings(id)
)
""")
conn.commit()

# Purana data clear karo (agar re-run kar rahe ho to duplicates na banein)
cursor.execute("DELETE FROM posting_skills")
conn.commit()

# ---- Har posting ka description padho, skills nikaalo, insert karo ----
cursor.execute("SELECT id, description FROM postings")
all_postings = cursor.fetchall()

rows_inserted = 0
postings_with_no_skills = 0

for posting_id, description in all_postings:
    skills_found = extract_skills(description)
    if not skills_found:
        postings_with_no_skills += 1
    for skill in skills_found:
        cursor.execute(
            "INSERT OR IGNORE INTO posting_skills (posting_id, skill) VALUES (?, ?)",
            (posting_id, skill)
        )
        rows_inserted += 1

conn.commit()

print(f"Total postings processed: {len(all_postings)}")
print(f"Total skill-mentions inserted: {rows_inserted}")
print(f"Postings with zero matched skills: {postings_with_no_skills}")

# ---- Quick check: top skills by frequency ----
cursor.execute("""
    SELECT skill, COUNT(*) as freq
    FROM posting_skills
    GROUP BY skill
    ORDER BY freq DESC
""")
print("\nTop skills across all postings:")
for skill, freq in cursor.fetchall():
    print(f"  {skill}: {freq}")

Total postings processed: 452
Total skill-mentions inserted: 348
Postings with zero matched skills: 296

Top skills across all postings:
  SQL: 72
  Python: 34
  Excel: 32
  Power BI: 31
  ETL: 28
  Tableau: 25
  Statistics: 25
  Big Data: 19
  Machine Learning: 17
  Data Warehousing: 11
  Azure: 9
  AWS: 8
  Communication: 7
  A/B Testing: 6
  SAS: 5
  Looker: 5
  GCP: 5
  Google Analytics: 4
  VBA: 2
  PowerPoint: 2
  R: 1


In [16]:

# ---- Data limitation ko DB me hi document kar dete hain (report ke liye) ----
cursor.execute("""
CREATE TABLE IF NOT EXISTS data_notes (
    note_key TEXT PRIMARY KEY,
    note_text TEXT
)
""")
cursor.execute("""
    INSERT OR REPLACE INTO data_notes (note_key, note_text)
    VALUES (?, ?)
""", (
    "description_truncation",
    "Adzuna's free-tier API returns truncated job descriptions (~500 characters), "
    "not the full posting text. As a result, skill-keyword extraction from description "
    "text represents an under-count, not a complete list of required skills per posting. "
    "296 of 452 postings (65%) had no skill keywords detected in the available snippet. "
    "Findings should be interpreted as directional signal from partial text, not exhaustive "
    "requirement lists."
))
conn.commit()
print("Limitation documented in data_notes table.\n")

# ---- City-wise skill demand ----
print("=" * 50)
print("SKILL DEMAND BY CITY")
print("=" * 50)

cursor.execute("""
    SELECT p.city_queried, ps.skill, COUNT(*) as freq
    FROM postings p
    JOIN posting_skills ps ON p.id = ps.posting_id
    GROUP BY p.city_queried, ps.skill
    ORDER BY p.city_queried, freq DESC
""")

from collections import defaultdict
city_skills = defaultdict(list)
for city, skill, freq in cursor.fetchall():
    city_skills[city].append((skill, freq))

for city, skills in city_skills.items():
    print(f"\n{city}:")
    for skill, freq in skills[:5]:  # top 5 per city
        print(f"  {skill}: {freq}")

# ---- Postings count per city (for context - denominator matters) ----
print("\n" + "=" * 50)
print("TOTAL POSTINGS PER CITY (for context)")
print("=" * 50)
cursor.execute("""
    SELECT city_queried, COUNT(*) as total
    FROM postings
    GROUP BY city_queried
    ORDER BY total DESC
""")
for city, total in cursor.fetchall():
    print(f"  {city}: {total}")

Limitation documented in data_notes table.

SKILL DEMAND BY CITY

Bangalore:
  SQL: 24
  Python: 15
  Excel: 13
  Power BI: 11
  Tableau: 8

Delhi NCR:
  SQL: 8
  Statistics: 5
  Python: 4
  ETL: 4
  Excel: 3

Hyderabad:
  SQL: 13
  ETL: 9
  Python: 4
  Excel: 4
  Tableau: 3

Mumbai:
  Power BI: 10
  Excel: 8
  SQL: 7
  Machine Learning: 5
  ETL: 5

Pune:
  SQL: 20
  ETL: 10
  Python: 9
  Tableau: 8
  Statistics: 7

TOTAL POSTINGS PER CITY (for context)
  Pune: 100
  Mumbai: 100
  Hyderabad: 100
  Bangalore: 100
  Delhi NCR: 52


In [18]:
# ---- Step A: Kitni postings me salary data actually hai? ----
cursor.execute("""
    SELECT COUNT(*) FROM postings
    WHERE salary_min IS NOT NULL OR salary_max IS NOT NULL
""")
postings_with_salary = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM postings")
total_postings = cursor.fetchone()[0]

print(f"Postings with salary data: {postings_with_salary} / {total_postings} "
      f"({postings_with_salary/total_postings*100:.1f}%)")

# ---- Step B: Agar data hai, see salary range ----
if postings_with_salary > 0:
    cursor.execute("""
        SELECT MIN(salary_min), MAX(salary_max), AVG((salary_min + salary_max) / 2.0)
        FROM postings
        WHERE salary_min IS NOT NULL AND salary_max IS NOT NULL
    """)
    min_sal, max_sal, avg_sal = cursor.fetchone()
    print(f"\nSalary range found: {min_sal} to {max_sal}")
    print(f"Average midpoint salary: {avg_sal}")

    # ---- Step C: Skill vs average salary (sirf agar kaafi data ho) ----
    print("\n" + "=" * 50)
    print("AVERAGE SALARY BY SKILL (where salary data exists)")
    print("=" * 50)

    cursor.execute("""
        SELECT ps.skill,
               COUNT(*) as num_postings,
               AVG((p.salary_min + p.salary_max) / 2.0) as avg_salary
        FROM postings p
        JOIN posting_skills ps ON p.id = ps.posting_id
        WHERE p.salary_min IS NOT NULL AND p.salary_max IS NOT NULL
        GROUP BY ps.skill
        HAVING COUNT(*) >= 3   -- kam se kam 3 postings honi chahiye, warna avg meaningless hai
        ORDER BY avg_salary DESC
    """)
    rows = cursor.fetchall()
    if rows:
        for skill, count, avg_salary in rows:
            print(f"  {skill}: avg ~{avg_salary:,.0f} (based on {count} postings)")
    else:
        print("  Not enough postings per skill (min 3 required) to compute reliable averages.")
else:
    print("\nNo salary data available in this dataset.")
    print("This is common for Indian job postings on Adzuna — many employers don't disclose salary.")
    print("This will be documented as a limitation; salary analysis will be skipped or noted as N/A.")

# ---- Step D: Ise bhi limitation ke roop me document karo agar kam data hai ----
if total_postings > 0:
    salary_pct = postings_with_salary / total_postings * 100
    if salary_pct < 20:
        cursor.execute("""
            INSERT OR REPLACE INTO data_notes (note_key, note_text)
            VALUES (?, ?)
        """, (
            "salary_data_sparse",
            f"Only {postings_with_salary} of {total_postings} postings ({salary_pct:.1f}%) "
            f"included salary data. This is typical for the Indian job market, where salary "
            f"disclosure is uncommon. Skill-vs-salary conclusions in this project should be "
            f"treated as indicative only, not statistically robust, given the small sample size."
        ))
        conn.commit()
        print(f"\nNote: Salary data is sparse ({salary_pct:.1f}%) - documented as a limitation.")

Postings with salary data: 104 / 452 (23.0%)

Salary range found: 0.0 to 4500000.0
Average midpoint salary: 1031067.3076923077

AVERAGE SALARY BY SKILL (where salary data exists)
  Google Analytics: avg ~2,312,500 (based on 4 postings)
  A/B Testing: avg ~1,712,500 (based on 4 postings)
  ETL: avg ~1,491,667 (based on 6 postings)
  Machine Learning: avg ~1,450,000 (based on 4 postings)
  Big Data: avg ~1,316,667 (based on 6 postings)
  Statistics: avg ~1,268,889 (based on 9 postings)
  Data Warehousing: avg ~1,225,000 (based on 4 postings)
  SQL: avg ~1,223,448 (based on 29 postings)
  Tableau: avg ~1,160,000 (based on 4 postings)
  Power BI: avg ~1,111,250 (based on 8 postings)
  Excel: avg ~1,106,154 (based on 13 postings)
  Python: avg ~971,538 (based on 13 postings)
  Communication: avg ~650,000 (based on 3 postings)


In [19]:
# Cell 8 - Zero-salary outliers 

cursor.execute("""
    SELECT id, title, company, salary_min, salary_max
    FROM postings
    WHERE salary_min = 0 OR salary_max = 0
""")
zero_salary_rows = cursor.fetchall()

print(f"Postings with salary_min or salary_max = 0: {len(zero_salary_rows)}")
for row in zero_salary_rows:
    print(f"  ID {row[0]}: {row[1]} at {row[2]} -> min={row[3]}, max={row[4]}")

Postings with salary_min or salary_max = 0: 14
  ID 4344578568: Data Analyst at Jobdost -> min=0.0, max=500000.0
  ID 2572832371: Data Analyst at BitClass -> min=0.0, max=1600000.0
  ID 2635526340: Data Analyst at Amazon -> min=0.0, max=800000.0
  ID 2177266370: Data Analyst at Marktine -> min=0.0, max=1500000.0
  ID 2653476396: Data Analyst at CIEL HR Services -> min=0.0, max=1500000.0
  ID 1993882092: Data Analyst at SveltetechTechnologies Pvt Ltd -> min=0.0, max=300000.0
  ID 2570069948: Data Analyst- Marketing at Unnati -> min=0.0, max=1500000.0
  ID 1993882129: Fullstack Developer at SveltetechTechnologies Pvt Ltd -> min=0.0, max=600000.0
  ID 2133622627: Fullstack Developer- Python at Bondburry Recruitmenrt Pvt Ltd -> min=0.0, max=1400000.0
  ID 2449398766: Data Analyst at Syrencloud -> min=0.0, max=800000.0
  ID 2344120568: PHP Developer at Yagna Technologies Pvt Ltd -> min=0.0, max=300000.0
  ID 2433361449: Data Analyst at MSMEx -> min=0.0, max=1200000.0
  ID 2398472037: Data A

In [20]:
# Cell 9 - Corrected salary analysis (zero-min cases fix)

print("=" * 50)
print("AVERAGE SALARY BY SKILL (corrected for zero-min cases)")
print("=" * 50)

cursor.execute("""
    SELECT ps.skill,
           COUNT(*) as num_postings,
           AVG(
               CASE
                   WHEN p.salary_min = 0 THEN p.salary_max
                   ELSE (p.salary_min + p.salary_max) / 2.0
               END
           ) as avg_salary
    FROM postings p
    JOIN posting_skills ps ON p.id = ps.posting_id
    WHERE p.salary_min IS NOT NULL AND p.salary_max IS NOT NULL
    GROUP BY ps.skill
    HAVING COUNT(*) >= 3
    ORDER BY avg_salary DESC
""")

for skill, count, avg_salary in cursor.fetchall():
    print(f"  {skill}: avg ~₹{avg_salary:,.0f} (based on {count} postings)")

# Note: 0-min cases documented as a handled data-quality adjustment
cursor.execute("""
    INSERT OR REPLACE INTO data_notes (note_key, note_text)
    VALUES (?, ?)
""", (
    "zero_min_salary_handling",
    "14 of 104 salaried postings had salary_min=0 (Adzuna's default when only a max was "
    "specified by the employer). These were treated as salary_max-only figures rather than "
    "averaged with 0, to avoid artificially deflating skill-salary averages."
))
conn.commit()
print("\nData-quality adjustment documented in data_notes.")

AVERAGE SALARY BY SKILL (corrected for zero-min cases)
  Google Analytics: avg ~₹2,312,500 (based on 4 postings)
  A/B Testing: avg ~₹1,712,500 (based on 4 postings)
  ETL: avg ~₹1,491,667 (based on 6 postings)
  Machine Learning: avg ~₹1,450,000 (based on 4 postings)
  Tableau: avg ~₹1,447,500 (based on 4 postings)
  Statistics: avg ~₹1,374,444 (based on 9 postings)
  Big Data: avg ~₹1,316,667 (based on 6 postings)
  SQL: avg ~₹1,276,897 (based on 29 postings)
  Power BI: avg ~₹1,255,000 (based on 8 postings)
  Data Warehousing: avg ~₹1,225,000 (based on 4 postings)
  Excel: avg ~₹1,190,769 (based on 13 postings)
  Python: avg ~₹1,002,308 (based on 13 postings)
  Communication: avg ~₹650,000 (based on 3 postings)

Data-quality adjustment documented in data_notes.


In [23]:
# Cell 10 - Personal Skill-Gap Analysis

# ---- put your current skills here ----
MY_SKILLS = {"SQL", "Python", "Excel", "Power BI", "ETL", "Statistics", "Data Cleaning"}

# ---- Creat table (to reuse in dashboard) ----
cursor.execute("""
CREATE TABLE IF NOT EXISTS my_skills (
    skill TEXT PRIMARY KEY
)
""")
cursor.execute("DELETE FROM my_skills")
for skill in MY_SKILLS:
    cursor.execute("INSERT INTO my_skills (skill) VALUES (?)", (skill,))
conn.commit()

# ---- Market demand: skill -> (num postings, avg salary if exists) ----
cursor.execute("""
    SELECT ps.skill,
           COUNT(DISTINCT ps.posting_id) as demand_count,
           AVG(
               CASE
                   WHEN p.salary_min = 0 THEN p.salary_max
                   WHEN p.salary_min IS NOT NULL AND p.salary_max IS NOT NULL
                        THEN (p.salary_min + p.salary_max) / 2.0
                   ELSE NULL
               END
           ) as avg_salary
    FROM posting_skills ps
    LEFT JOIN postings p ON ps.posting_id = p.id
    GROUP BY ps.skill
    ORDER BY demand_count DESC
""")
market_data = cursor.fetchall()  # list of (skill, demand_count, avg_salary)
market_skills = {row[0] for row in market_data}
market_lookup = {row[0]: (row[1], row[2]) for row in market_data}

# ---- Create three categories  ----
matched = MY_SKILLS & market_skills
gaps = market_skills - MY_SKILLS
extras = MY_SKILLS - market_skills

print("=" * 55)
print("✅ MATCHED SKILLS (you have + market wants)")
print("=" * 55)
for skill in sorted(matched, key=lambda s: -market_lookup[s][0]):
    count, avg_sal = market_lookup[skill]
    sal_str = f"₹{avg_sal:,.0f}" if avg_sal else "no salary data"
    print(f"  {skill}: {count} postings mention this, avg salary {sal_str}")

print("\n" + "=" * 55)
print("⚠️  SKILL GAPS (market wants, you don't have yet) — PRIORITY LIST")
print("=" * 55)
gap_sorted = sorted(gaps, key=lambda s: -market_lookup[s][0])
for skill in gap_sorted:
    count, avg_sal = market_lookup[skill]
    sal_str = f"₹{avg_sal:,.0f}" if avg_sal else "no salary data"
    print(f"  {skill}: {count} postings mention this, avg salary {sal_str}")

print("\n" + "=" * 55)
print("💡 YOUR EXTRAS (you have, low/no market signal in this dataset)")
print("=" * 55)
if extras:
    for skill in extras:
        print(f"  {skill}: not detected in current dataset's descriptions")
else:
    print("  None — all your skills showed up in market demand data.")

# ---- Simple readiness score ----
if market_skills:
    readiness_pct = len(matched) / len(market_skills) * 100
    print(f"\n📊 Rough readiness score: {len(matched)}/{len(market_skills)} in-demand skills covered "
          f"({readiness_pct:.0f}%)")
    print("   (Note: this is based on frequency of mention in truncated descriptions, "
          "not a weighted importance score — treat as directional.)")

✅ MATCHED SKILLS (you have + market wants)
  SQL: 72 postings mention this, avg salary ₹1,276,897
  Python: 34 postings mention this, avg salary ₹1,002,308
  Excel: 32 postings mention this, avg salary ₹1,190,769
  Power BI: 31 postings mention this, avg salary ₹1,255,000
  ETL: 28 postings mention this, avg salary ₹1,491,667
  Statistics: 25 postings mention this, avg salary ₹1,374,444

⚠️  SKILL GAPS (market wants, you don't have yet) — PRIORITY LIST
  Tableau: 25 postings mention this, avg salary ₹1,447,500
  Big Data: 19 postings mention this, avg salary ₹1,316,667
  Machine Learning: 17 postings mention this, avg salary ₹1,450,000
  Data Warehousing: 11 postings mention this, avg salary ₹1,225,000
  Azure: 9 postings mention this, avg salary ₹1,400,000
  AWS: 8 postings mention this, avg salary ₹1,050,000
  Communication: 7 postings mention this, avg salary ₹650,000
  A/B Testing: 6 postings mention this, avg salary ₹1,712,500
  Looker: 5 postings mention this, avg salary no salar